# Market Narrative Pipeline — Colab quickstart
Runtime → GPU (T4 is enough). Add Colab secrets: `MINIO_ENDPOINT`, `MINIO_ACCESS_KEY`, `MINIO_SECRET_KEY`, `FMP_API_KEY` (recommended), optional `BENZINGA_API_KEY`, `HF_TOKEN`.

In [ ]:
import os, sys, subprocess
BRANCH = 'claude/quirky-wozniak-72igbq'
REPO = '/content/OpenBB'
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: print('FAILED:', cmd, '\n', r.stderr[-500:])
    return r.stdout.strip()
if not os.path.isdir(REPO + '/.git'):
    sh(f'git clone -q --depth 1 --single-branch -b {BRANCH} https://github.com/jam2205/OpenBB.git {REPO}')
else:
    # Force the checkout to match GitHub exactly (discards local edits inside the clone).
    sh(f'git -C {REPO} fetch -q --depth 1 origin {BRANCH}')
    sh(f'git -C {REPO} reset -q --hard FETCH_HEAD')
print('Code version on disk:', sh(f'git -C {REPO} log -1 --format="%h %s"'))
%cd /content/OpenBB/pipelines/market_narrative
# Drop any already-imported copy so the next import uses the code on disk (no restart needed).
stale = [m for m in sys.modules if m == 'mnp' or m.startswith('mnp.')]
for m in stale: del sys.modules[m]
print(f'Cleared {len(stale)} cached pipeline modules')
!pip -q install -r requirements.txt 2>&1 | grep -v "dependency resolver\|incompatible" | tail -2

In [ ]:
import os
from google.colab import userdata
# Canonical name -> names it may be saved under in Colab Secrets (first found wins). Add yours if different.
ALIASES = {
    "FRED_API_KEY": ["FRED_API_KEY", "FRED_KEY", "API_FRED_KEY", "fred_api_key"],
    "FMP_API_KEY": ["FMP_API_KEY", "FMP_KEY", "fmp_api_key"],
    "BENZINGA_API_KEY": ["BENZINGA_API_KEY", "BENZINGA_KEY"],
    "HF_TOKEN": ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HF_API_KEY"],
    "MINIO_ENDPOINT": ["MINIO_ENDPOINT", "MINIO_URL"],
    "MINIO_ACCESS_KEY": ["MINIO_ACCESS_KEY", "MINIO_ROOT_USER"],
    "MINIO_SECRET_KEY": ["MINIO_SECRET_KEY", "MINIO_ROOT_PASSWORD"],
}
for canon, names in ALIASES.items():
    for n in names:
        try:
            v = userdata.get(n)
        except Exception:
            continue
        if v:
            os.environ[canon] = v
            break
from openbb import obb
for env, attr in [("FRED_API_KEY", "fred_api_key"), ("FMP_API_KEY", "fmp_api_key"), ("BENZINGA_API_KEY", "benzinga_api_key")]:
    if os.environ.get(env):
        setattr(obb.user.credentials, attr, os.environ[env])
print({k: bool(os.environ.get(k)) for k in ALIASES})

## 1. AcyLa predictive model: skill table + live signals
LightGBM-only by default. Signals with test dir. accuracy < 55% show as "no skill".

In [ ]:
from datetime import date, timedelta
from mnp.predictive import AcyLaModel
from mnp import sources
m = AcyLaModel('AcyLa/multi-asset-predictive-model')
start = date.today() - timedelta(days=900)
m.predict_many({t: sources.ohlcv(t, start, 'yfinance') for t in ['SPY', 'QQQ', 'GLD']})

## 2. Smoke-test the pre-trained NLP models (no training needed)

In [ ]:
from mnp.nlp import NLP
from mnp.run import load_config
import pandas as pd
cfg = load_config(); nlp = NLP(cfg['models'])
news = pd.DataFrame({'text': ['Gold rallies as yields fall after soft CPI', 'Nvidia guidance disappoints',
                              'BoJ hints at yen intervention', 'Treasury auction tails, 10Y jumps 8bp']})
nlp.attribute_drivers(news, cfg['drivers'])[['text','tone','signed','driver','driver_sim','driver_method']]

## 3. Full run → regime-specific report

In [ ]:
import logging; logging.basicConfig(level=logging.INFO)
from mnp.run import run
path = run(out_dir='reports')
from IPython.display import Markdown; Markdown(open(path).read())

## IMF (no key): find indicator symbols, then paste them into `config.yaml → imf.indicators`

In [ ]:
from mnp import sources
res = sources.imf_search('exchange rate')
res.head(30)  # copy the `symbol` values you want (format DATAFLOW::INDICATOR)

## 4. (Optional, later) fine-tune ModernBERT-large heads
Only when you have labelled headlines and want more accuracy. Saved heads in `heads/` are picked up automatically.
```
!python -m mnp.finetune driver --csv data/headlines.csv --out heads/driver
!python -m mnp.finetune tone   --csv data/headlines.csv --out heads/tone --max-length 2048
```